# 🎓 Classroom Quality Monitoring — Qwen2.5-VL Fine-Tuning on Google Colab

This notebook fine-tunes **Qwen2.5-VL-7B-Instruct** using QLoRA (4-bit quantization) on a free Google Colab GPU (T4/A100).

### ⚡ Quick Steps:
1. Open this notebook in Google Colab: **Runtime → Change runtime type → T4 GPU**.
2. Upload `data.zip` (located at `fine_tuning/data.zip` in your project).
3. Run all cells below.
4. Download `lora_adapter.zip` when complete!

### Step 1: Check NVIDIA GPU Availability

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies

In [ ]:
!pip install -q -U torch torchvision transformers peft bitsandbytes accelerate datasets trl httpx pillow

### Step 3: Unzip & Load Classroom Dataset

In [ ]:
import os
import zipfile

if os.path.exists("data.zip"):
    print("📦 Unzipping data.zip...")
    with zipfile.ZipFile("data.zip", 'r') as zip_ref:
        zip_ref.extractall("data")
    print("✅ Dataset unzipped to ./data/")
else:
    print("⚠️ data.zip not found! Upload fine_tuning/data.zip to Colab files sidebar.")

### Step 4: Run QLoRA Fine-Tuning on GPU

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {model_id} onto GPU...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

# Configure LoRA Adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Save trained adapter
model.save_pretrained("./lora_adapter")
print("🎉 LoRA Fine-Tuning Complete! Adapter saved to ./lora_adapter")

### Step 5: Zip & Download Trained Weights

In [ ]:
from google.colab import files
!zip -r lora_adapter.zip ./lora_adapter
files.download("lora_adapter.zip")